# Query Generation Agent

In [90]:
import os
import json
from typing import List, Tuple
from dotenv import load_dotenv
from openai import OpenAI

In [91]:
load_dotenv()

True

In [76]:
system_message = """
    You specialize in crafting best and most efficient google search operators that work well.
    The prompt you generate will be used as queries for Exa and SerpAPI web search later on.
""".strip()

In [78]:
user_prompt = """
Generate search queries for my job search.

Requirements:
- I need a remote AI engineering role focused on agentic systems, AI llm_modules, RAG pipelines, LoRA/QLoRA fine-tuning, AI integration, AI-driven applications, and LLM engineering including frontier and open-source Hugging Face models.
- The role must be remote-friendly for someone working from South Korea.
- Junior, mid-level, internship are preferred. Senior roles are still acceptable if realistic for a 3-year-experience developer.
- First 10 queries must be formatted for the google search operator style for SERP queries, and the next 10 queries are sematic search queries for Exa query style 
- Output exactly this JSON schema:

{
  "serp": ["... exactly 10 strings ..."],
  "exa": ["... exactly 10 strings ..."]
}

Rules:
- Return only raw JSON.
- No markdown fences.
- No explanation.
- Each list must contain exactly 10 strings.
""".strip()

In [79]:
llm = OpenAI()

In [80]:
response = llm.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]
)

In [86]:
print(response.choices[0].message.content)

{
  "serp": [
    "intitle:\"AI Engineer\" (agent OR agentic OR \"AI agent\" OR \"agentic systems\") (RAG OR \"retrieval augmented\" OR \"retrieval-augmented\") (LoRA OR QLoRA OR \"fine-tuning\" OR \"fine tuning\") \"remote\" (\"South Korea\" OR Korea OR \"work from South Korea\") (junior OR \"entry-level\" OR intern OR \"mid-level\")",
    "site:linkedin.com/jobs intitle:\"Machine Learning Engineer\" (agent OR agentic OR \"AI agent\") \"remote\" (\"South Korea\" OR Korea) (LoRA OR QLoRA OR RAG OR \"Hugging Face\" OR \"LLM engineering\") (junior OR mid OR intern)",
    "site:remoteok.io \"AI engineer\" (RAG OR \"retrieval augmented\" OR LoRA OR QLoRA OR \"fine-tuning\") \"remote\" (Korea OR \"South Korea\") (junior OR \"entry-level\" OR mid)",
    "site:angel.co/jobs \"AI\" (agent OR \"AI agent\" OR \"agentic systems\") \"remote\" (intern OR junior OR \"entry level\" OR mid) \"Hugging Face\"",
    "\"AI agents\" OR agentic \"remote\" \"South Korea\" (junior OR intern OR \"mid-level\") 

In [92]:
llm_response = response.choices[0].message.content

In [93]:
def process_response() -> Tuple[List[str], List[str]]:
    """
    load the LLM JSON response and separate each key (ser, exa) for easier process on SerpAPI and Exa individually.
    """

    try:
        data = json.loads(llm_response)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse LLM JSON response: {e}\n")

    serp = data.get("serp")
    exa = data.get("exa")
    
    if not isinstance(serp, list) or not isinstance(exa, list): 
        raise ValueError(f"Response must contain 'serp' and 'exa' as lists.\nRaw response: {llm_res}")

    if len(serp) != 10 or len(exa) != 10:
        raise ValueError(
            f"Expected exactly 10 serp and 10 exa queries, got serp={len(serp)}, exa={len(exa)}."
        )

    return serp, exa

In [96]:
# Final serach queries in a list for SerpAPI
queries = process_response()

queries[0]

['intitle:"AI Engineer" (agent OR agentic OR "AI agent" OR "agentic systems") (RAG OR "retrieval augmented" OR "retrieval-augmented") (LoRA OR QLoRA OR "fine-tuning" OR "fine tuning") "remote" ("South Korea" OR Korea OR "work from South Korea") (junior OR "entry-level" OR intern OR "mid-level")',
 'site:linkedin.com/jobs intitle:"Machine Learning Engineer" (agent OR agentic OR "AI agent") "remote" ("South Korea" OR Korea) (LoRA OR QLoRA OR RAG OR "Hugging Face" OR "LLM engineering") (junior OR mid OR intern)',
 'site:remoteok.io "AI engineer" (RAG OR "retrieval augmented" OR LoRA OR QLoRA OR "fine-tuning") "remote" (Korea OR "South Korea") (junior OR "entry-level" OR mid)',
 'site:angel.co/jobs "AI" (agent OR "AI agent" OR "agentic systems") "remote" (intern OR junior OR "entry level" OR mid) "Hugging Face"',
 '"AI agents" OR agentic "remote" "South Korea" (junior OR intern OR "mid-level") -senior -lead',
 'intitle:"LLM Engineer" OR intitle:"LLM Engineering" (open-source OR "Hugging Fa

In [73]:
def extract_serp_queries(q: Tuple[List[str], List[str]]) -> List[str]:
    """
    Extract the SerpAPI query list out of all queries
    """

    return queries[0]

def extract_exa_queries(q: Tuple[List[str], List[str]]) -> List[str]:
    """
    Extract the Exa query list out of all queries
    """

    return queries[1]

In [ ]:
extract_serp_queries(queries)

In [ ]:
extract_exa_queries(queries)